# Bolt #6: Residual Analysis & Reporting

**Objective**: Analyze the Hybrid Baseline residuals to identify performance bottlenecks and validate statistical assumptions.

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import scipy.stats as stats
from statsmodels.stats.stattools import durbin_watson

# Configuration
RUN_PATH = Path("../../artifacts/runs/dry_run_v1_retry3/")
RAW_DATA_PATH = Path("../../data/raw/")

print(f"Analyzing run: {RUN_PATH.resolve().name}")

## 1. Data Loading & Enrichment

Joining OOF residuals with Oil prices and Holiday events.

In [ ]:
# Load OOF Residuals
df_oof = pd.read_csv(RUN_PATH / "oof_residuals.csv", parse_dates=["date"])

# Load Oil
df_oil = pd.read_csv(RAW_DATA_PATH / "oil.csv", parse_dates=["date"])
# Axiom: Forward-fill oil prices (weekends/holidays)
df_oil = df_oil.set_index("date").resample('D').ffill().reset_index()

# Load Holidays
df_holidays = pd.read_csv(RAW_DATA_PATH / "holidays_events.csv", parse_dates=["date"])

# Merge
df = df_oof.merge(df_oil, on="date", how="left")
df = df.merge(df_holidays, on="date", how="left")

print(f"Enriched Data Shape: {df.shape}")
df.head()

## 2. Segment Metrics & Macro Performance

Identifying high-level failure patterns across the Store-Family matrix.

In [ ]:
# Calculate RMSLE per segment
agg_metrics = df.groupby(['store_nbr', 'family']).agg({
    'sq_log_error': 'mean',
    'sales': ['sum', 'mean'],
    'residual': 'std'
}).reset_index()

agg_metrics.columns = ['store_nbr', 'family', 'rmsle_sq', 'total_sales', 'avg_sales', 'residual_std']
agg_metrics['rmsle'] = np.sqrt(agg_metrics['rmsle_sq'])

print(f"Average Segment RMSLE: {agg_metrics['rmsle'].mean():.4f}")
agg_metrics.sort_values('rmsle', ascending=False).head(10)

In [ ]:
# Macro Heatmap: Matrix of RMSLE
fig = px.density_heatmap(
    agg_metrics, 
    x="family", 
    y="store_nbr", 
    z="rmsle",
    title="Global RMSLE Heatmap (Store vs Family)",
    labels={'rmsle': 'RMSLE'},
    color_continuous_scale="Viridis",
    nbinsy=agg_metrics['store_nbr'].nunique()
)
fig.update_layout(xaxis={'categoryorder':'total descending'})
fig.show()

## 3. Statistical Residual Analysis

Validating if the residuals follow theoretical assumptions (Normality, Independence).

In [ ]:
# Normality Tests
residuals_sample = df['residual'].dropna().sample(min(5000, len(df)))
k2, p_k2 = stats.normaltest(residuals_sample)

print("--- Normality Tests ---")
print(f"D'Agostino's K^2 p-value: {p_k2:.4e}")

fig_dist = px.histogram(df, x="residual", marginal="box", title="Distribution of Residuals")
fig_dist.show()

In [ ]:
# Autocorrelation (Durbin-Watson)
dw_stat = durbin_watson(df['residual'].dropna())
print(f"Durbin-Watson Statistic: {dw_stat:.4f}")

## 4. Blind Spot Deep-Dives (Top 5)

Visualizing the segments where the model struggles the most.

In [ ]:
def plot_segment_deepdive(store_nbr, family):
    subset = df[(df['store_nbr'] == store_nbr) & (df['family'] == family)].sort_values('date')
    
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True, 
                        vertical_spacing=0.1, subplot_titles=(f"Sales vs Pred: Store {store_nbr}, {family}", "Residuals"))
    
    # Row 1: Sales
    fig.add_trace(go.Scatter(x=subset['date'], y=subset['sales'], name="Actual", line=dict(color='blue')), row=1, col=1)
    fig.add_trace(go.Scatter(x=subset['date'], y=subset['sales_pred'], name="Predicted", line=dict(color='orange', dash='dash')), row=1, col=1)
    
    # Row 2: Residuals
    fig.add_trace(go.Bar(x=subset['date'], y=subset['residual'], name="Residual", marker_color='red'), row=2, col=1)
    
    # Highlight Holidays
    holidays_in_period = subset[subset['type'].notnull()]
    for _, h in holidays_in_period.iterrows():
        fig.add_vline(x=h['date'], line_width=1, line_dash="dot", line_color="green", row='all', col=1)
        
    fig.update_layout(height=600, showlegend=True, title_text=f"Deep Dive: Segment {store_nbr}-{family}")
    return fig

top_5_worst = agg_metrics.sort_values('rmsle', ascending=False).head(5)

for _, row in top_5_worst.iterrows():
    fig = plot_segment_deepdive(row['store_nbr'], row['family'])
    fig.show()

## 5. Actionable Insights & Recommendations

Summary of findings from this analysis:

1. **Residual Distribution**: Observe if residuals are zero-centered. Significant skewness might suggest target transformation issues.
2. **Autocorrelation**: DW statistic < 1.5 indicates missing temporal signals (lags).
3. **Segment clusters**: Heatmap clusters reveal if failures are store-specific or family-specific.